In [43]:
import os
import glob
import shutil
import logging
import traceback
import numpy as np
import pandas as pd
import talib
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
from utils.config import TWStockConfig, MarketDateRange
from utils.data_utils import MarketDataProcessor

In [44]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('stock_data_processing.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)

In [45]:
def process_price_data(df):
    """
    處理價格數據，包含資料清理和格式轉換
    """
    try:
        # 複製DataFrame避免修改原始資料
        df = df.copy()
        
        # 處理價格欄位的千分位逗號
        price_columns = ['開盤價', '最高價', '最低價', '收盤價', '最後揭示買價', '最後揭示賣價']
        for col in price_columns:
            if col in df.columns:
                df[col] = df[col].apply(lambda x: str(x).replace(',', '') if isinstance(x, str) else x)
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # 處理成交量欄位
        volume_columns = ['成交股數', '成交筆數', '成交金額']
        for col in volume_columns:
            if col in df.columns:
                df[col] = df[col].apply(lambda x: str(x).replace(',', '') if isinstance(x, str) else x)
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # 處理日期欄位
        if '日期' in df.columns:
            df['日期'] = pd.to_datetime(df['日期']).dt.strftime('%Y-%m-%d')
            
        # 處理特殊字符
        df = df.replace('--', np.nan)
        df = df.replace('', np.nan)
        
        return df
        
    except Exception as e:
        logging.error(f"Error in process_price_data: {e}")
        logging.error(traceback.format_exc())
        return None

In [46]:
def validate_config():
    """
    驗證必要的目錄和檔案是否存在
    """
    base_path = "D:\\Min\\Python\\Project\\FA_Data"
    required_dirs = [
        "meta_data",
        "technical_analysis",
        "meta_data\\backup"
    ]
    
    try:
        for dir_path in required_dirs:
            full_path = os.path.join(base_path, dir_path)
            if not os.path.exists(full_path):
                os.makedirs(full_path)
                logging.info(f"Created directory: {full_path}")
                
        return True
    except Exception as e:
        logging.error(f"Error validating config: {e}")
        return False

In [47]:
def calculate_and_store_indicators(df, stock_id=None, date=None, by_date=False):
    """
    Calculate technical indicators and save to files
    """
    try:
        # 數據預處理
        df = process_price_data(df)
        if df is None:
            return False
            
        # 檢查數據有效性
        price_columns = ['開盤價', '最高價', '最低價', '收盤價']
        if df[price_columns].isnull().all().any():
            logging.warning(f"Missing price data for {'date '+date if by_date else 'stock ID '+stock_id}")
            return False
            
        # 排序並重設索引
        df = df.sort_values('日期').reset_index(drop=True)
        
        # 使用收盤價計算技術指標
        close_prices = df['收盤價'].values
        high_prices = df['最高價'].values
        low_prices = df['最低價'].values
        
        # 計算技術指標
        df['SMA30'] = talib.SMA(close_prices, timeperiod=30)
        df['DEMA30'] = talib.DEMA(close_prices, timeperiod=30)
        df['EMA30'] = talib.EMA(close_prices, timeperiod=30)
        df['RSI'] = talib.RSI(close_prices, timeperiod=14)
        
        macd, signal, hist = talib.MACD(close_prices, 
                                      fastperiod=12, 
                                      slowperiod=26, 
                                      signalperiod=9)
        df['MACD'] = macd
        df['MACD_signal'] = signal
        df['MACD_hist'] = hist
        
        # 計算KD指標
        if not df[['最高價', '最低價', '收盤價']].isnull().any().any():
            slowk, slowd = talib.STOCH(high_prices, 
                                     low_prices, 
                                     close_prices,
                                     fastk_period=5, 
                                     slowk_period=3, 
                                     slowk_matype=0,
                                     slowd_period=3, 
                                     slowd_matype=0)
            df['slowk'] = slowk
            df['slowd'] = slowd
        
        df['TSF'] = talib.TSF(close_prices, timeperiod=14)
        upper, middle, lower = talib.BBANDS(close_prices, 
                                          timeperiod=30, 
                                          nbdevup=2, 
                                          nbdevdn=2, 
                                          matype=0)
        df['middleband'] = middle
        
        if not df[['最高價', '最低價']].isnull().any().any():
            df['SAR'] = talib.SAR(high_prices, 
                                 low_prices, 
                                 acceleration=0.02, 
                                 maximum=0.2)

        # 儲存結果
        base_path = "D:\\Min\\Python\\Project\\FA_Data"
        save_path = f"{base_path}\\technical_analysis\\{stock_id}_indicators.csv"
        
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        df.to_csv(save_path, index=False, encoding='utf-8-sig')
        
        return df
        
    except Exception as e:
        logging.error(f"Error processing stock {stock_id}: {e}")
        logging.error(traceback.format_exc())
        return None

In [48]:
def process_stock_data_batch():
    """批次處理股票資料"""
    try:
        base_path = "D:\\Min\\Python\\Project\\FA_Data"
        stock_data_whole = pd.read_csv(f"{base_path}\\meta_data\\stock_data_whole.csv", low_memory=False)
        
        all_processed_data = []
        grouped = stock_data_whole.groupby('證券代號')
        total_stocks = len(grouped)
        
        processed_count = 0
        failed_stocks = []
        
        # 使用tqdm包裝grouped.items()
        progress_bar = tqdm(grouped, total=total_stocks, 
                          desc="Processing stocks", 
                          unit="stock",
                          bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]')
        
        for stock_id, group_df in progress_bar:
            result_df = calculate_and_store_indicators(group_df.copy(), stock_id=stock_id)
            if isinstance(result_df, pd.DataFrame):
                all_processed_data.append(result_df)
                processed_count += 1
                progress_bar.set_postfix({'Success': f"{processed_count}/{total_stocks}"})
            else:
                failed_stocks.append(stock_id)
                progress_bar.set_postfix({'Failed': stock_id})

        progress_bar.close()

        if all_processed_data:
            # 合併所有處理後的資料
            logging.info("Merging processed data...")
            all_stocks_df = pd.concat(all_processed_data, ignore_index=True)
            
            # 備份和儲存
            today = datetime.now().strftime('%Y%m%d')
            backup_dir = f"{base_path}\\meta_data\\backup"
            backup_path = f"{backup_dir}\\all_stocks_data_{today}.csv"
            
            # 保存新的合併檔案
            logging.info("Saving merged data...")
            all_stocks_df.to_csv(f"{base_path}\\meta_data\\all_stocks_data.csv", 
                               index=False, encoding='utf-8-sig')
            
            # 建立備份
            shutil.copy2(f"{base_path}\\meta_data\\all_stocks_data.csv", backup_path)
            
            # 清理舊備份（保留最近7天）
            backup_files = glob.glob(f"{backup_dir}\\all_stocks_data_*.csv")
            backup_files.sort(reverse=True)
            for old_backup in backup_files[7:]:
                os.remove(old_backup)
            
            # 輸出處理結果統計
            logging.info(f"Processing completed:")
            logging.info(f"- Successfully processed: {processed_count}/{total_stocks} stocks")
            if failed_stocks:
                logging.info(f"- Failed stocks: {', '.join(failed_stocks)}")
            
            return True
        else:
            logging.warning("No data was successfully processed")
            return False
            
    except Exception as e:
        logging.error(f"Error in batch processing: {e}")
        logging.error(traceback.format_exc())
        return False

In [49]:
def main():
    """主程序"""
    try:
        config = TWStockConfig()
        date_range = MarketDateRange.last_n_days(45)
        
        logging.info("Starting batch processing...")
        if process_stock_data_batch():
            logging.info("Batch processing completed successfully")
            return True
        else:
            logging.error("Batch processing failed")
            return False
            
    except Exception as e:
        logging.error(f"Error in main process: {e}")
        logging.error(traceback.format_exc())
        return False

In [50]:
if __name__ == "__main__":
    if validate_config():
        success = main()
        if success:
            logging.info("Process completed successfully")
            
            # 顯示一個股票的處理結果作為範例
            example_stock = "2330"
            try:
                df = pd.read_csv(f"D:\\Min\\Python\\Project\\FA_Data\\technical_analysis\\{example_stock}_indicators.csv")
                print(f"\n{example_stock} 技術指標計算結果範例:")
                print(df.tail())
            except Exception as e:
                logging.error(f"Error showing example results: {e}")
    else:
        logging.error("Configuration validation failed")

2024-11-01 00:19:41,359 - INFO - Starting batch processing...
Processing stocks: 100%|████████████████████████████████████████████████████████| 1148/1148 [01:03<00:00, 18.21stock/s]
2024-11-01 00:20:48,623 - INFO - Merging processed data...
2024-11-01 00:20:49,400 - INFO - Saving merged data...
2024-11-01 00:21:29,441 - INFO - Processing completed:
2024-11-01 00:21:29,442 - INFO - - Successfully processed: 1147/1148 stocks
2024-11-01 00:21:29,442 - INFO - - Failed stocks: 911201
2024-11-01 00:21:29,688 - INFO - Batch processing completed successfully
2024-11-01 00:21:29,689 - INFO - Process completed successfully



2330 技術指標計算結果範例:
      證券代號 證券名稱      成交股數   成交筆數         成交金額     開盤價     最高價     最低價     收盤價  \
2580  2330  台積電  40791484  42113  43477824029  1070.0  1075.0  1055.0  1060.0   
2581  2330  台積電  23347890  25884  24867260319  1065.0  1070.0  1060.0  1065.0   
2582  2330  台積電  41665065  55098  44216316045  1075.0  1080.0  1050.0  1050.0   
2583  2330  台積電  48097705  93617  49680152085  1035.0  1040.0  1020.0  1040.0   
2584  2330  台積電  40765247  46750  42368799669  1040.0  1055.0  1030.0  1030.0   

     漲跌(+/-)  ...        EMA30        RSI       MACD  MACD_signal  MACD_hist  \
2580     NaN  ...  1013.694551  60.763201  29.450019    27.403697   2.046322   
2581       +  ...  1017.004580  61.748523  28.522309    27.627420   0.894890   
2582       -  ...  1019.133317  57.114713  26.273849    27.356706  -1.082856   
2583       -  ...  1020.479555  54.194846  23.415097    26.568384  -3.153287   
2584       -  ...  1021.093777  51.366826  20.110775    25.276862  -5.166087   

          slow